# 20_pairwise_interactions.ipynb

In [ ]:
# Import appropriate configuration information
from pathlib import Path
path = Path.cwd().parent.parent
HYPERPARAM_DEST = path / "notebooks" / "modeling"
DATA = path / "data" / "raw" / "imagingFeatures.csv"
TARGET = 'ER'

A ranked list of candidate pairwise interactions that XGB thinks matter for ER (repeat for PR, HER2)

In [ ]:
import shap
import xgboost as xgb
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import json

# Fetch json object that will contain the optimal hyperparameters
with open(HYPERPARAM_DEST / "ER" / "ALL_IMG.json", "r") as f:
    results = json.load(f)

# Find the XGBoost entry
xgb_entry = next(m for m in results if m["model"] == "XGBoost")
xgb_best_params = xgb_entry["best_params"]

# Retrieve the data the model was trained on
clinical_data_path = path / "data" / "raw" / "clinicalData_clean.csv"
features = pd.read_csv(DATA)
features.rename(columns={'Patient.ID': 'Patient ID'}, inplace=True, errors='ignore')
clin = pd.read_csv(clinical_data_path)
data = features.merge(clin[['Patient ID', TARGET]], on='Patient ID', how='inner')
data = data.drop('Unnamed: 0', axis=1, errors='ignore').dropna() # Won't run if there are NA values
y = data[TARGET]
le = LabelEncoder()
y = le.fit_transform(y.astype(str))
X = data.drop([TARGET, 'Patient ID'], axis=1, errors='ignore') # same features used in training

# XGB model already fit on your favorite feature set
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', **xgb_best_params)
xgb_model.fit(X, y)

explainer = shap.TreeExplainer(xgb_model) # returns (n_samples, n_features, n_features)

interaction_vals = explainer.shap_interaction_values(X)

# Mean |SHAP interaction| across samples
import numpy as np
mean_int = np.abs(interaction_vals).mean(axis=0)
np.fill_diagonal(mean_int, 0)              # drop main-effect diagonal
top_pairs = (
    pd.DataFrame(mean_int, index=X.columns, columns=X.columns)
      .stack()
      .sort_values(ascending=False)
      .head(50)                            # top 50 is usually plenty
)
top_pairs.to_csv("top_interactions_xgb.csv")
print(top_pairs.head(10))


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [15:45:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
